# Reward Model Loss — 面试版

**难度：** Medium

**面试要点：** 数值稳定版本的 sigmoid，防止 exp(-margin) 溢出

In [ ]:
import torch

In [ ]:
# ✅ INTERVIEW

def reward_model_loss(chosen_hidden, rejected_hidden, reward_head):
    r_chosen = (chosen_hidden @ reward_head).squeeze(-1)
    r_rejected = (rejected_hidden @ reward_head).squeeze(-1)
    margin = r_chosen - r_rejected

    # ---- 数值稳定的 -log(sigmoid(x)) ----
    # 直接计算 log(1/(1+exp(-x))) 在 x 很大或很小时会有问题:
    #   x >> 0: exp(-x) ≈ 0, 1+0 = 1, log(1) = 0 ✓
    #   x << 0: exp(-x) → ∞, 1+∞ = ∞, 1/∞ → 0, log(0) → -∞ ✗ 数值问题！
    #
    # 数学恒等式: -log(σ(x)) = -log(1/(1+exp(-x))) = log(1+exp(-x))
    #   = log(1 + exp(-x))
    #   = softplus(-x)
    #
    # 但这在 x >> 0 时 exp(-x) 下溢为 0，不过 log(1+0)=0 没问题
    # 在 x << 0 时 exp(-x) 溢出！
    #
    # 最终方案: 分段计算
    #   x >= 0: log(1 + exp(-x))  — exp(-x) 不会溢出
    #   x <  0: -x + log(1 + exp(x))  — exp(x) 不会溢出
    # 这就是 softplus 的数值稳定实现
    loss = torch.where(
        margin >= 0,
        torch.log(1.0 + torch.exp(-margin)),
        -margin + torch.log(1.0 + torch.exp(margin))
    )
    return loss.mean()

In [ ]:
import math
torch.manual_seed(0)

B, D = 8, 64
reward_head = torch.randn(D, 1)
h = torch.randn(B, D)

loss = reward_model_loss(h, h, reward_head)
print(f"Equal hidden states => loss = {loss.item():.4f}  (expected ≈ {math.log(2):.4f})")

# 极端情况: margin 很大或很小
chosen_extreme = torch.ones(B, D) * 100
rejected_extreme = -torch.ones(B, D) * 100
loss_extreme = reward_model_loss(chosen_extreme, rejected_extreme, reward_head)
print(f"Extreme margin => loss = {loss_extreme.item():.6f}  (expected ≈ 0)")

In [ ]:
from torch_judge import check
check("reward_model")

## 📝 核心思路总结

1. **数值稳定 sigmoid**：分段计算 `log(1+exp(-x))`，避免溢出
2. **torch.where 分支**：x≥0 用 `log(1+exp(-x))`，x<0 用 `-x+log(1+exp(x))`
3. **本质是 softplus**：`-log(σ(x)) = softplus(-x)`